# Genixcure PacBio — Circular Start Normalization & Exact Sequence Count

**목표:** EcoRV 처리 전 원형(plasmid)의 시작 좌표는 임의적일 수 있으므로, 실제 매핑된 PacBio 5,115,819 Reads를 **Reference 동일 방향 + 동일 시작점**으로 정규화한 뒤 정확히 같은 전체 서열을 합산합니다.

- 원본: `PLA_6_13.hifi_reads.bam` 전체 5,140,518 Reads (122.7GB), `Alignment_FULL_BAM_v1/chunks`의 완성된 515개 매핑 결과 재사용
- Reference: 6,848 bp EcoRV-linear FASTA; **원본 원형 플라스미드 1,269 bp**에 해당하는 고유한 31-mer를 모두의 시작으로 사용
- **같은 서열이 어느 위치에서 시작해도**, `+/-` 방향 및 원형 회전을 통일합니다. 삽입·결실·UMI/Barcode는 바꾸지 않습니다.
- 결과: `mapped_circular_start_normalized_count.csv.gz` (sequence, count, full_span_count, other_mapped_count, length_bp, sequence_sha256)
- 파일럿 12,000개와 달리 **전체 BAM**을 읽습니다. 복구 가능한 10,000 Reads 체크포인트를 작성합니다.
- 원본 실험은 **EcoRV로 linearized 된 DNA**입니다. 이 원형 rotation은 계산상의 동등성 검사이며 실제 원형 DNA였다는 증거가 아닙니다.
- **시작 앵커가 없거나 두 번 이상 나온 Read**는 억지로 뭉치지 않고 unresolved로 따로 기록합니다. `count` 합계는 정규화에 성공한 mapped Reads를 의미합니다.


In [ ]:
import subprocess, sys
try:
    import pysam
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pysam>=0.22,<0.24'], check=True)
    import pysam
print('pysam', pysam.__version__)


In [ ]:
from pathlib import Path
from google.colab import drive, auth
import google.auth
from googleapiclient.discovery import build
import json

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
api = build('drive', 'v3', credentials=creds, cache_discovery=False)

ROOT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13')
REF_DIR=ROOT/'Reference_EcoRV_Linear'
REF=REF_DIR/'plasmid_EcoRV_linear_minimap2.fasta'
MAPPING=REF_DIR/'Alignment_FULL_BAM_v1'/'chunks'
SCRIPT=REF_DIR/'Genixcure_PacBio_Circular_Start_Count.py'
OUT=REF_DIR/'Alignment_Circular_Normalized_Count_v1'
BAM=ROOT/'PLA_6_13.hifi_reads.bam'
SOURCE_BAM_ID='1RthcVb15hJpLF3V_P9vtgDCE1q9yH-Ln'

assert REF.is_file() and SCRIPT.is_file(), 'Reference or script missing from Drive folder'
assert (MAPPING/'chunk_00514.done.json').is_file(), 'Full mapped 515-chunk results missing'
if not BAM.is_file():
    name='PLA_6_13.hifi_reads.bam'
    # We cannot rely on filesystem folder names as Drive folder IDs; use the known source parent ID.
    parent_id='12IFMSn8buE2A4GN0TC68Yv3cDl3abRCV'
    found=api.files().list(q=f"'{parent_id}' in parents and name = '{name}' and trashed = false",fields='files(id,mimeType,shortcutDetails)',pageSize=30,supportsAllDrives=True,includeItemsFromAllDrives=True).execute().get('files',[])
    if not any(x.get('shortcutDetails',{}).get('targetId')==SOURCE_BAM_ID for x in found):
        shortcut=api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.shortcut', 'parents':[parent_id], 'shortcutDetails':{'targetId':SOURCE_BAM_ID}}, fields='id,name').execute()
        print('BAM shortcut:',shortcut)
assert BAM.is_file(), 'Original BAM not mounted. Confirm MyDrive shortcut and reconnect mount.'
assert BAM.stat().st_size==122715338770, 'BAM source size mismatch'
OUT.mkdir(parents=True, exist_ok=True)
print('INPUT:',BAM, 'bytes:',BAM.stat().st_size)
print('MAPPING:',MAPPING)
print('OUTPUT:',OUT)


## 1. 원형 시작점 정규화 — 전수 515개 Batch

각 Batch는 기존 원본 BAM BGZF virtual offset을 이용해 10,000개를 정확히 다시 읽고, 저장된 `mapped` / `strand`와 read 이름을 교차 검증합니다.
`*.normalized.done.json`이 마지막에 저장되므로, 연결이 끊겨도 Colab 재실행으로 완료 Batch부터 이어서 진행합니다.


In [ ]:
subprocess.run([sys.executable, str(SCRIPT), '--self-test'], check=True)


In [ ]:
# 기본값 0 = 완료된 515개 Batch 전체. 재실행하면 기존 완료 Batch는 건너뜁니다.
MAX_NEW_CHUNKS=0
cmd=[sys.executable, '-u', str(SCRIPT),
    '--bam',str(BAM),'--reference',str(REF),'--mapping-chunks',str(MAPPING),
    '--out',str(OUT),'--mapping-chunk-reads','10000',
    '--max-chunks',str(MAX_NEW_CHUNKS),'--stage','normalize']
print('Starting full BAM circular start normalization')
subprocess.run(cmd,check=True)


## 2. 정규화된 전체 서열 Exact Count 집계

전체 515개 normalized chunk를 정확히 합산합니다. 임시 SQLite 데이터베이스는 Colab 로컬 디스크에 만들며, 원본 시퀀스나 BAM을 로컬에 전부 복제하지 않습니다.
- 전체 표: `mapped_circular_start_normalized_count.csv.gz` (서열별 Count)
- TOP100: `top_100_mapped_circular_start_counts.csv`
- 검증: `circular_start_normalized_summary.json`

동일한 시작 31-mer가 존재하지 않는 Read는 **정확한 좌표로 회전할 수 없어 미분류**됩니다. 별도 `chunks/chunk_XXXXX.unresolved.csv.gz`에 보존됩니다.


In [ ]:
import shutil
scratch=Path('/content/genixcure_circular_aggregation')
scratch.mkdir(parents=True,exist_ok=True)
print('Scratch free GB:',round(shutil.disk_usage(scratch).free/1e9,1))
cmd=[sys.executable, '-u', str(SCRIPT),
    '--reference',str(REF),'--mapping-chunks',str(MAPPING),'--out',str(OUT),
    '--scratch',str(scratch),'--stage','aggregate']
subprocess.run(cmd,check=True)


In [ ]:
import pandas as pd
from IPython.display import display
summary_path=OUT/'circular_start_normalized_summary.json'
if summary_path.exists():
    summary=json.loads(summary_path.read_text())
    print({k:summary[k] for k in ['status','all_bam_reads','all_mapped_reads','normalized_mapped_reads','normalized_full_span_reads','distinct_circular_normalized_sequences','unresolved_mapped_reads']})
    print('Reference common first 31-mer:',summary['anchor_sequence'])
    top=OUT/'top_100_mapped_circular_start_counts.csv'
    if top.exists():
        t=pd.read_csv(top)
        t['sequence_prefix_60bp']=t['sequence'].str[:60]
        display(t[['count','full_span_count','length_bp','sequence_prefix_60bp']].head(20))
    print('FULL TABLE:',OUT/'mapped_circular_start_normalized_count.csv.gz')
else:
    print('Normalization has not completed. See chunks directory; rerun from above to resume.')
